# Notebook 03: Static No-Arbitrage Surface Constraints
A valid volatility surface must be free from static arbitrage:
1. **Calendar Spread Arbitrage**: Total variance $w(k, T)$ must be non-decreasing in $T$ for all $k$.
2. **Butterfly Arbitrage**: The implied probability density must be non-negative everywhere, which is equivalent to Gatheral-Jacquier condition $g(k) \ge 0$.

This notebook demonstrates the execution of the static no-arbitrage checks on both valid and invalid surfaces.


In [ ]:
import numpy as np
import sys
sys.path.append('../src')

from dpp.calibration.svi import get_svi_surface
from dpp.arbitrage.checks import calendar_arbitrage_violations, butterfly_arbitrage_violations

# Expiries and SVI parameters
T_expiries = np.array([0.25, 0.5, 1.0])
k_grid = np.linspace(-0.5, 0.5, 21)


In [ ]:
# Case 1: Arbitrage-Free SVI surface
# Parameters: a, b, rho, m, sigma
svi_clean = np.array([
    [0.04, 0.1, -0.2, 0.0, 0.1],  # T = 0.25
    [0.05, 0.1, -0.2, 0.0, 0.1],  # T = 0.50
    [0.06, 0.1, -0.2, 0.0, 0.1]   # T = 1.00
])
w_surface_clean = get_svi_surface(T_expiries, svi_clean)

cal_viol_clean = calendar_arbitrage_violations(w_surface_clean, k_grid, T_expiries)
but_viol_clean = butterfly_arbitrage_violations(w_surface_clean, k_grid, T_expiries)

print("Arbitrage-Free Surface Verification:")
print("  Calendar Spread Violations:", len(cal_viol_clean))
print("  Butterfly Violations:      ", len(but_viol_clean))


In [ ]:
# Case 2: Surface with deliberate Calendar Spread Arbitrage
# Variance decreases from T=0.25 to T=0.50
svi_cal_arb = np.array([
    [0.05, 0.1, -0.2, 0.0, 0.1],  # T = 0.25
    [0.03, 0.1, -0.2, 0.0, 0.1],  # T = 0.50 (Variance dropped!)
    [0.06, 0.1, -0.2, 0.0, 0.1]   # T = 1.00
])
w_surface_cal = get_svi_surface(T_expiries, svi_cal_arb)
cal_violations = calendar_arbitrage_violations(w_surface_cal, k_grid, T_expiries)

print("Calendar Arbitrage Surface Verification:")
print(f"  Calendar Spread Violations detected: {len(cal_violations)}")
if cal_violations:
    print(f"  Example violation at k={cal_violations[0][0]:.2f}: T_prev={cal_violations[0][1]} -> T_curr={cal_violations[0][2]}")


In [ ]:
# Case 3: Surface with deliberate Butterfly Arbitrage
# High wing slope parameter b causes negative density
svi_but_arb = np.array([
    [0.04, 0.8, -0.9, 0.0, 0.01],  # T = 0.25 (Very steep smile!)
    [0.05, 0.1, -0.2, 0.0, 0.1],   # T = 0.50
    [0.06, 0.1, -0.2, 0.0, 0.1]    # T = 1.00
])
w_surface_but = get_svi_surface(T_expiries, svi_but_arb)
but_violations = butterfly_arbitrage_violations(w_surface_but, k_grid, T_expiries)

print("Butterfly Arbitrage Surface Verification:")
print(f"  Butterfly Violations detected: {len(but_violations)}")
if but_violations:
    print(f"  Example violation at k={but_violations[0][0]:.2f}, T={but_violations[0][1]:.2f}: g_val={but_violations[0][2]:.4f}")
